In [ ]:
LOG_DIR = '../../experiments/current/'


In [ ]:
# Parameters
LOG_DIR = "../../experiments/current/"


In [ ]:
import pandas as pd
import json
import os


def load_jsonl(path):
    """Charge un fichier de JSON objets concaténés (format multi-lignes ou NDJSON)."""
    with open(path) as f:
        content = f.read()
    if not content.strip():
        return pd.DataFrame()
    decoder = json.JSONDecoder()
    records, pos = [], 0
    while pos < len(content):
        try:
            obj, end = decoder.raw_decode(content, pos)
            records.append(obj)
            pos = end
            while pos < len(content) and content[pos] in ' \n\t\r':
                pos += 1
        except json.JSONDecodeError:
            break
    df = pd.DataFrame(records)
    if not df.empty and 'time' in df.columns:
        df['time'] = pd.to_datetime(df['time'])
    return df

# overwrite with actual folder

df_success = load_jsonl(LOG_DIR + 'llm_exchanges.jsonl')
df_error   = load_jsonl(LOG_DIR + 'llm_errors.jsonl')
#df_success = load_jsonl('../../experiments/2026-05-15_12_26/llm_exchanges.jsonl')
#df_error   = load_jsonl('../../experiments/2026-05-15_12_26/llm_errors.jsonl')

print(f"Succès : {len(df_success)} | Erreurs : {len(df_error)}")
df_success.head()

def _img_path(title, ext="png"):
    name = "".join(c if c.isalnum() or c in "-_" else "_" for c in title)
    img_dir = os.path.join(LOG_DIR, "images/llm_traffic")
    os.makedirs(img_dir, exist_ok=True)
    return os.path.join(img_dir, f"{name}.{ext}")


In [ ]:
df_success.columns

df_freq = df_success[['time', 'provider']]

df_freq

## Appels par minute par provider (succès / erreurs)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# Combiner succès et erreurs avec statut
df_s = df_success[['time', 'provider']].copy()
df_s['status'] = 'success'

df_e = df_error[['time', 'provider']].copy()
df_e['status'] = 'error'

df_all = pd.concat([df_s, df_e], ignore_index=True)
df_all['minute'] = df_all['time'].dt.floor('min')

# Compter les appels par minute, provider et statut
calls_per_min = (
    df_all.groupby(['minute', 'provider', 'status'])
    .size()
    .unstack('status', fill_value=0)
)

providers_sorted = sorted(df_all['provider'].unique())
n = len(providers_sorted)

fig, axes = plt.subplots(n, 1, figsize=(14, 3 * n), sharex=True)
if n == 1:
    axes = [axes]

for ax, prov in zip(axes, providers_sorted):
    if prov in calls_per_min.index.get_level_values('provider'):
        prov_data = calls_per_min.xs(prov, level='provider')
        if 'success' in prov_data.columns:
            ax.plot(prov_data.index, prov_data['success'], label='succès', color='steelblue', linewidth=1.5)
        if 'error' in prov_data.columns:
            ax.plot(prov_data.index, prov_data['error'], label='erreur', color='tomato', linestyle='--', linewidth=1.5)
    ax.set_title(prov, fontsize=10, fontweight='bold')
    ax.set_ylabel('appels/min')
    ax.legend(fontsize=8)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
    ax.grid(True, alpha=0.3)

plt.savefig(_img_path("provider_calls"), dpi=200)

plt.xlabel('Heure')
plt.tight_layout()
plt.show()


## Taux de réussite par provider

In [ ]:
total_calls = df_all.groupby('provider').size().rename('total')
success_counts = df_all[df_all['status'] == 'success'].groupby('provider').size().rename('success')
error_counts = df_all[df_all['status'] == 'error'].groupby('provider').size().rename('errors')

# Max d'appels sur une minute (toutes minutes confondues, success + erreurs)
calls_per_min_total = (
    df_all.groupby(['provider', 'minute'])
    .size()
)
max_calls_per_min = calls_per_min_total.groupby('provider').max().rename('max_appels/min')

summary = pd.concat([total_calls, success_counts, error_counts, max_calls_per_min], axis=1).fillna(0)
summary[['total', 'success', 'errors', 'max_appels/min']] = summary[['total', 'success', 'errors', 'max_appels/min']].astype(int)
summary['taux_réussite'] = (summary['success'] / summary['total'] * 100).round(1)
summary = summary.sort_values('taux_réussite')

# Graphique
fig, ax = plt.subplots(figsize=(10, max(4, len(summary) * 0.7)))

colors = ['green' if v >= 80 else 'orange' if v >= 50 else 'tomato' for v in summary['taux_réussite']]
bars = ax.barh(summary.index, summary['taux_réussite'], color=colors, edgecolor='white')

for bar, val in zip(bars, summary['taux_réussite']):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height() / 2,
            f'{val:.1f}%', va='center', fontsize=9)

ax.set_xlabel('Taux de réussite (%)')
ax.set_title('Taux de réussite par provider')
ax.set_xlim(0, 110)
ax.axvline(x=100, color='gray', linestyle='--', alpha=0.4)
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig(_img_path("provider_success_rate"), dpi=200)
plt.show()

# Tableau récapitulatif
summary.sort_values('taux_réussite', ascending=False)


## Principales erreurs par provider

In [ ]:
summary_errors = pd.DataFrame(
    df_error.groupby(['provider', 'error_type', 'error_message'])
    .size()
    .sort_values(ascending=False)
    .reset_index(name='count')
)

summary_errors = summary_errors[summary_errors['count'] > 1]

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', None)
display(summary_errors.head(20))
